# Структурная проверка README — Ось 1, версия 2 (исправленная)

Этот notebook — переработанная **структурная ось** (бывшие «39 критериев»). Он:
- **чинит** сломанные критерии (читабельность 2.4.7, детектор примеров 2.4.6, честные `check_method`);
- **разжалует** мёртвые/всегда-проходящие проверки в preflight-гейт;
- **добавляет** новые ОБЪЕКТИВНЫЕ проверки целостности, которые реально ловят дефекты
  (сломанные таблицы, дословные повторы, диаграммы не по теме, оборванные фразы);
- показывает **before/after** на реальном документе.

Важная граница: здесь только то, что можно проверить **скриптом объективно** (есть/нет, число, целостность).
Градуированное качество подачи (насколько пример хорош, связно ли изложено) живёт на **дидактической оси**
(ноутбук с жюри). Не пытаемся ловить вкус регэкспом.

Запускается полностью **без LLM** — все проверки скриптовые.


## Что изменилось (карта правок)

| Критерий | Было | Стало |
|---|---|---|
| **2.4.7** читабельность | нормировка `[0,30]→[50,80]` делала условие `50≤x≤80` **всегда истинным** → вечная 1 | честная формула Флеша-**Оборневой** для русского + реальная полоса; критерий может упасть |
| **2.4.6** пример/кейс | `"пример" in text` ловит «примерно», «применять» → почти всегда 1; помечен `AI_AGENT` | поиск по **границам слова** + явный маркер `**Пример**`; честный `SCRIPT` |
| **2.4.4** определения | помечен `AI_AGENT`, внутри регэксп; премирует шаблон `**X** — это …` | честный `SCRIPT`, переведён в `SOFT` (форма, не суть) |
| **1.1–1.6** структура | считались в общий балл, почти всегда =1 | **preflight-гейт** «не сломан», вынесены из балла качества |
| дубль логики читабельности | две копии в `check()` и `check_document()` | одна функция |
| пороги | `thresholds.py` ≠ CRITERIA.md (аннотация 220-520 vs 300-800) | один источник правды в этом ноутбуке |
| **NEW** целостность таблиц | — | строки таблиц не слиты с прозой, число колонок согласовано |
| **NEW** дословные повторы | — | repetition-ratio + почти-дубли предложений ниже порога |
| **NEW** диаграмма ↔ тема | — | каждый mermaid связан с заголовком своего раздела |
| **NEW** оборванные фразы | — | сбалансированы «…», нет осиротевших » и обрывков |
| **NEW** единство имени проекта | — | в теле один project-id |


## Настройки и загрузка документа

In [1]:
import re, statistics
from pathlib import Path
from dataclasses import dataclass, field
from collections import Counter
import os

# Единый источник правды по порогам (синхронизирован, без расхождения с докой)
THRESHOLDS = {
    "annotation_chars": (220, 800),
    "intro_words": (70, 180),
    "instruction_words": (80, 250),
    "theory_parts": (3, 7),
    "theory_words_per_part": (110, 260),
    "practice_tasks_range": (2, 8),
    # новые пороги целостности
    "readability_band": (45, 80),       # Флеш-Оборнева; ПРОВИЗОРНО, калибруется на данных
    "repetition_ratio_max": 0.06,       # доля повторяющихся 8-грамм
    "near_dup_max": 4,                  # почти-дублирующих пар предложений
    "diagram_topic_min": 0.15,          # Jaccard nodes<->heading
}

def find_doc():
    for base in ["/mnt/user-data/uploads", ".", "/mnt/data"]:
        p = Path(base)
        if p.exists():
            hits = sorted(p.glob("*.md"))
            if hits: return hits[0]
    return None
DOC_PATH = Path(os.environ["DOC_PATH"]) if os.environ.get("DOC_PATH") else find_doc()
assert DOC_PATH and DOC_PATH.exists(), "Не найден .md документ."
MD = DOC_PATH.read_text(encoding="utf-8")
print("Документ:", DOC_PATH.name, f"({len(MD)} символов)")

Документ: PjM15_PublicSpeaking.md (22594 символов)


## Текстовые утилиты

In [2]:
def clean_prose(t):
    """Убирает code/mermaid, таблицы, html, markdown-разметку — оставляет прозу для подсчётов."""
    t = re.sub(r"```.*?```", " ", t, flags=re.S)
    t = re.sub(r"^\|.*\|.*$", " ", t, flags=re.M)
    t = re.sub(r"<[^>]+>", " ", t)
    t = re.sub(r"[#*_>`\[\]()]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

def words(t): return re.findall(r"[А-Яа-яЁёA-Za-z]+", t)
def count_words(t): return len(words(clean_prose(t)))
def sentences(t):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", t) if len(s.strip()) > 3]

def syllables_ru(w): return max(1, sum(1 for ch in w.lower() if ch in "аеёиоуыэюя"))

def oborneva_fre(text):
    """Флеш-Оборнева (адаптация Reading Ease под русский). Выше = легче."""
    s = sentences(text); w = words(text)
    if not s or not w: return None
    asl = len(w) / len(s)
    asw = sum(syllables_ru(x) for x in w) / len(w)
    return 206.835 - 1.3 * asl - 60.1 * asw

# Блоки документа
def chapter(md, n):
    m = re.search(rf"^##\s+Глава\s+{n}[^\n]*\n(.*?)(?=^##\s+Глава\s+{n+1}|^##\s+Бонус|^##\s+Заключение|\Z)",
                  md, flags=re.S|re.M)
    return m.group(1).strip() if m else ""

def theory_parts(md):
    ch2 = chapter(md, 2)
    out = []
    for m in re.finditer(r"^###\s+(2\.\d+\.[^\n]*)\n(.*?)(?=^###\s+|\Z)", ch2, flags=re.S|re.M):
        out.append((m.group(1).strip(), m.group(2).strip()))
    return out

print("Глав найдено:", sum(1 for n in (1,2,3) if chapter(MD,n)))
print("Частей теории:", len(theory_parts(MD)))

Глав найдено: 3
Частей теории: 3


## Модель результата критерия (честные метки)

In [3]:
@dataclass
class Crit:
    id: str
    title: str
    method: str          # SCRIPT / SBERT / AI_AGENT — ЧЕСТНО, как реально считается
    group: str           # preflight / kept / fixed / new
    strictness: str      # HARD / SOFT
    score: int = 0       # 0/1
    comments: list = field(default_factory=list)
    details: dict = field(default_factory=dict)

REGISTRY = []   # сюда складываем результаты
def emit(**kw):
    c = Crit(**kw); REGISTRY.append(c); return c

## PREFLIGHT (структура не сломана) — бывшие 1.1–1.6, вынесены из балла качества

In [4]:
def run_preflight(md):
    res = []
    res.append(emit(id="P.1", title="H1 в первой строке", method="SCRIPT", group="preflight",
        strictness="HARD", score=1 if md.lstrip().startswith("# ") else 0,
        comments=[] if md.lstrip().startswith("# ") else ["Нет H1 в первой строке"]))
    # аннотация: текст после H1 до первого ## без заголовков/списков
    m = re.search(r"^#\s+.+$", md, re.M)
    ann = ""
    if m:
        ann = md[m.end():].split("\n## ", 1)[0].strip()
    ok = bool(ann) and not re.search(r"^##", ann, re.M) and not re.search(r"^\s*[-*+]", ann, re.M)
    res.append(emit(id="P.2", title="Блок аннотации после H1", method="SCRIPT", group="preflight",
        strictness="HARD", score=1 if ok else 0, comments=[] if ok else ["Аннотация пуста/содержит заголовки или списки"],
        details={"annotation_chars": len(ann)}))
    toc = re.search(r"^##\s+(Содержание|Оглавление)\s*$", md, re.M)
    toc_lines = 0
    if toc:
        block = md[toc.end():].split("\n## ", 1)[0]
        toc_lines = len([l for l in block.splitlines() if l.strip()])
    res.append(emit(id="P.3", title="Оглавление (≥3 строк)", method="SCRIPT", group="preflight",
        strictness="HARD", score=1 if toc and toc_lines >= 3 else 0,
        comments=[] if toc and toc_lines>=3 else ["Нет оглавления или <3 строк"], details={"toc_lines": toc_lines}))
    for n, name, pid in [(1,"Глава 1 (введение)","P.4"),(2,"Глава 2 (теория)","P.5"),(3,"Глава 3 (практика)","P.6")]:
        c = chapter(md, n)
        res.append(emit(id=pid, title=f"{name} присутствует", method="SCRIPT", group="preflight",
            strictness="HARD", score=1 if len(c) > 50 else 0,
            comments=[] if len(c)>50 else [f"{name}: пусто/слишком коротко"]))
    return res
_ = run_preflight(MD)
print("preflight:", sum(c.score for c in REGISTRY), "/", len(REGISTRY))

preflight: 6 / 6


## KEPT — полезные оригинальные критерии (честно перемаркированы)

In [5]:
def run_kept(md):
    # аннотация по длине (синхронизированный порог)
    m = re.search(r"^#\s+.+$", md, re.M)
    ann = md[m.end():].split("\n## ", 1)[0].strip() if m else ""
    lo, hi = THRESHOLDS["annotation_chars"]
    emit(id="2.1.1", title="Длина аннотации", method="SCRIPT", group="kept", strictness="HARD",
         score=1 if lo <= len(ann) <= hi else 0,
         comments=[] if lo<=len(ann)<=hi else [f"{len(ann)} симв., норма {lo}-{hi}"], details={"chars": len(ann)})

    # 2.2.2 валидность ссылок оглавления (anchor -> заголовок)
    def slug(h):
        h = h.strip().lower()
        h = re.sub(r"[^\w\s-]", "", h, flags=re.U)
        return re.sub(r"\s+", "-", h)
    heads = {slug(h) for h in re.findall(r"^#{1,4}\s+(.+)$", md, re.M)}
    anchors = re.findall(r"\]\(#([^)]+)\)", md)
    bad = [a for a in anchors if a not in heads]
    emit(id="2.2.2", title="Ссылки оглавления ведут на заголовки", method="SCRIPT", group="kept",
         strictness="HARD", score=1 if anchors and not bad else 0,
         comments=[] if not bad else [f"битых якорей: {len(bad)} (напр. {bad[0]})"],
         details={"anchors": len(anchors), "broken": bad[:5]})

    # части теории: количество и объём
    parts = theory_parts(md)
    plo, phi = THRESHOLDS["theory_parts"]
    emit(id="2.4.1", title="Кол-во частей теории", method="SCRIPT", group="kept", strictness="HARD",
         score=1 if plo <= len(parts) <= phi else 0,
         comments=[] if plo<=len(parts)<=phi else [f"{len(parts)} частей, норма {plo}-{phi}"])
    wlo, whi = THRESHOLDS["theory_words_per_part"]
    bad_vol = [(t, count_words(b)) for t,b in parts if not (wlo <= count_words(b) <= whi)]
    emit(id="2.4.3", title="Объём каждой части теории", method="SCRIPT", group="kept", strictness="HARD",
         score=1 if parts and not bad_vol else 0,
         comments=[] if not bad_vol else [f"{len(bad_vol)} частей вне {wlo}-{whi} слов"],
         details={"per_part_words": [count_words(b) for _,b in parts]})

    # количество задач
    tasks = re.findall(r"^###\s+Задани[ея]\s+\d+\.", md, re.M)
    tlo, thi = THRESHOLDS["practice_tasks_range"]
    emit(id="2.5.1", title="Кол-во задач практики", method="SCRIPT", group="kept", strictness="HARD",
         score=1 if tlo <= len(tasks) <= thi else 0,
         comments=[] if tlo<=len(tasks)<=thi else [f"{len(tasks)} задач, норма {tlo}-{thi}"])

    # 2.5.5 у каждой задачи есть артефакт + путь
    ch3 = chapter(md, 3)
    blocks = re.split(r"^###\s+Задани[ея]\s+\d+\.", ch3, flags=re.M)[1:]
    no_path = sum(1 for b in blocks if not re.search(r"`[^`]+\.(md|csv|xlsx|png|pdf)`|/[\w\-./]+", b))
    emit(id="2.5.5", title="Артефакт и путь в каждой задаче", method="SCRIPT", group="kept", strictness="HARD",
         score=1 if blocks and no_path == 0 else 0,
         comments=[] if no_path==0 else [f"{no_path} задач без явного пути артефакта"])
run_kept(MD)
print("kept добавлены")

kept добавлены


## FIXED — исправленные критерии

In [6]:
def run_fixed(md):
    parts = theory_parts(md)

    # --- 2.4.6 пример/кейс: ИСПРАВЛЕНО (границы слова + явный маркер) ---
    def has_real_example(t):
        if re.search(r"\*\*\s*(Пример|Кейс|Ситуация)", t, re.I): return True
        return bool(re.search(r"\b(пример|примеры|кейс|кейсы|ситуация|ситуации)\b", t.lower()))
    miss = [title for title, body in parts if not has_real_example(body)]
    emit(id="2.4.6", title="Пример/кейс в каждой части (исправлено)", method="SCRIPT", group="fixed",
         strictness="HARD", score=1 if parts and not miss else 0,
         comments=[] if not miss else [f"без примера: {len(miss)} частей"],
         details={"note": "по границам слова, не подстрока; больше не ловит 'примерно'"})

    # --- 2.4.7 читабельность: ИСПРАВЛЕНО (Флеш-Оборнева, реальная полоса) ---
    band_lo, band_hi = THRESHOLDS["readability_band"]
    scores = []
    for title, body in parts:
        fre = oborneva_fre(clean_prose(body))
        if fre is not None: scores.append(round(fre, 1))
    avg = round(statistics.mean(scores), 1) if scores else 0
    ok = bool(scores) and band_lo <= avg <= band_hi
    emit(id="2.4.7", title="Читабельность (Флеш-Оборнева, исправлено)", method="SCRIPT", group="fixed",
         strictness="HARD", score=1 if ok else 0,
         comments=[] if ok else [f"средняя {avg}, полоса {band_lo}-{band_hi}"],
         details={"per_part_fre": scores, "avg": avg,
                  "note": "убрана тавтологичная нормировка [0,30]->[50,80]; теперь критерий может упасть"})

    # --- 2.4.4 определения: перемаркировано в SOFT, честный SCRIPT ---
    def has_definition(t):
        return bool(re.search(r"\*\*[^*]+\*\*\s*[—–-]\s*это|\*\*[^*]+\*\*\s*[—–-]\s+\w", t))
    no_def = [title for title, body in parts if not has_definition(body)]
    emit(id="2.4.4", title="Определения терминов (перемаркировано)", method="SCRIPT", group="fixed",
         strictness="SOFT", score=1 if parts and not no_def else 0,
         comments=[] if not no_def else [f"без явного определения: {len(no_def)}"],
         details={"note": "SOFT: проверяет форму '**X** — это...', не гарантирует осмысленность"})
run_fixed(MD)
print("fixed добавлены")

fixed добавлены


## NEW — новые объективные проверки целостности (ловят то, что не ловили 39)

In [7]:
def run_new(md):
    # N.1 целостность таблиц: строки не слиты с прозой + согласованность колонок
    long_rows = [ln[:90] for ln in md.splitlines() if ln.count("|") >= 2 and len(ln) > 200]
    # согласованность числа колонок внутри каждой таблицы
    col_issue = 0
    block = []
    def flush(block):
        nonlocal_issue = 0
        rows = [r for r in block if "|" in r and not re.match(r"^\s*\|?[\s:|-]+\|?\s*$", r)]
        counts = [r.count("|") for r in rows]
        return 1 if counts and (max(counts) - min(counts) > 1) else 0
    cur = []
    for ln in md.splitlines():
        if ln.strip().startswith("|"):
            cur.append(ln)
        else:
            if cur: col_issue += flush(cur); cur = []
    if cur: col_issue += flush(cur)
    bad = len(long_rows) + col_issue
    emit(id="N.1", title="Целостность таблиц", method="SCRIPT", group="new", strictness="HARD",
         score=1 if bad == 0 else 0,
         comments=[] if bad==0 else [f"{len(long_rows)} строк слиты с прозой, {col_issue} таблиц с рассогласованием колонок"],
         details={"merged_rows": long_rows[:3]})

    # N.2 дословные повторы (template bleed)
    def rep_ratio(t, n=8):
        w = re.findall(r"\w+", t.lower())
        g = [" ".join(w[i:i+n]) for i in range(len(w)-n+1)]
        if not g: return 0.0
        c = Counter(g); return sum(v for v in c.values() if v>1)/len(g)
    sents = sentences(clean_prose(md))
    sets = [set(re.findall(r"\w+", s.lower())) for s in sents]
    ndup = 0; ex = []
    for i in range(len(sets)):
        for j in range(i+1, len(sets)):
            if sets[i] and sets[j] and len(sets[i]&sets[j])/len(sets[i]|sets[j]) >= 0.7:
                ndup += 1
                if len(ex)<2: ex.append(sents[i][:70])
    rr = round(rep_ratio(md), 3)
    fail = rr > THRESHOLDS["repetition_ratio_max"] or ndup > THRESHOLDS["near_dup_max"]
    emit(id="N.2", title="Нет дословных повторов/болванок", method="SCRIPT", group="new", strictness="HARD",
         score=0 if fail else 1,
         comments=[f"repetition_ratio={rr} (макс {THRESHOLDS['repetition_ratio_max']}), почти-дублей={ndup} (макс {THRESHOLDS['near_dup_max']})"] if fail else [],
         details={"examples": ex})

    # N.3 диаграмма <-> тема раздела
    def nearest_heading(idx):
        h = None
        for m in re.finditer(r"^#{2,3}\s+(.+)$", md[:idx], re.M): h = m.group(1)
        return h
    mism = []
    for m in re.finditer(r"```mermaid(.*?)```", md, flags=re.S):
        head = nearest_heading(m.start())
        nodes = set(re.findall(r"[А-Яа-яЁё]{4,}", m.group(1).lower()))
        hw = set(re.findall(r"[А-Яа-яЁё]{4,}", (head or "").lower()))
        jac = len(nodes&hw)/len(nodes|hw) if (nodes|hw) else 1.0
        if jac < THRESHOLDS["diagram_topic_min"]: mism.append((head, round(jac,3)))
    emit(id="N.3", title="Диаграммы соответствуют теме раздела", method="SCRIPT", group="new", strictness="HARD",
         score=1 if not mism else 0,
         comments=[] if not mism else [f"{len(mism)} диаграмм вне темы (напр. '{mism[0][0]}' match={mism[0][1]})"],
         details={"note": "грубая эвристика по токенам; пограничные случаи -> на дидактическую ось"})

    # N.4 оборванные фразы: баланс «», осиротевшие »
    op, cl = md.count("«"), md.count("»")
    orphan = 0
    for m in re.finditer("»", md):
        if "«" not in md[max(0,m.start()-200):m.start()]: orphan += 1
    bad = (op != cl) or orphan > 0
    emit(id="N.4", title="Нет оборванных фраз/кавычек", method="SCRIPT", group="new", strictness="HARD",
         score=0 if bad else 1,
         comments=[f"«={op}, »={cl}, осиротевших »={orphan}"] if bad else [],
         details={})

    # N.5 единство имени проекта
    ids = sorted(set(re.findall(r"\b[A-Za-z]{2,}\d+_[A-Za-z0-9_]+\b", md)))
    emit(id="N.5", title="Единый идентификатор проекта", method="SCRIPT", group="new", strictness="SOFT",
         score=1 if len(ids) <= 1 else 0,
         comments=[] if len(ids)<=1 else [f"в теле несколько id: {ids}"], details={"ids": ids})
run_new(MD)
print("new добавлены")

new добавлены


## Итоговый отчёт по группам

In [8]:
GROUP_TITLE = {"preflight":"PREFLIGHT (структура не сломана, гейт)",
               "kept":"KEPT (полезные оригинальные)",
               "fixed":"FIXED (исправленные)",
               "new":"NEW (новые проверки целостности)"}
print("="*72); print("ДОКУМЕНТ:", DOC_PATH.name); print("="*72)
for g in ["preflight","kept","fixed","new"]:
    items = [c for c in REGISTRY if c.group == g]
    s = sum(c.score for c in items)
    print(f"\n### {GROUP_TITLE[g]}  —  {s}/{len(items)}")
    for c in items:
        mark = "✅" if c.score else "❌"
        soft = " (SOFT)" if c.strictness=="SOFT" else ""
        print(f"  {mark} {c.id:7} [{c.method:9}] {c.title}{soft}")
        for cm in c.comments: print(f"          ↳ {cm}")

hard_fail = [c for c in REGISTRY if c.score==0 and c.strictness=="HARD"]
print("\n" + "="*72)
print(f"HARD-провалов: {len(hard_fail)} -> {'ВОЗВРАТ на доработку/ревью' if hard_fail else 'структурно чисто'}")
print("Список:", ", ".join(c.id for c in hard_fail))

ДОКУМЕНТ: PjM15_PublicSpeaking.md

### PREFLIGHT (структура не сломана, гейт)  —  6/6
  ✅ P.1     [SCRIPT   ] H1 в первой строке
  ✅ P.2     [SCRIPT   ] Блок аннотации после H1
  ✅ P.3     [SCRIPT   ] Оглавление (≥3 строк)
  ✅ P.4     [SCRIPT   ] Глава 1 (введение) присутствует
  ✅ P.5     [SCRIPT   ] Глава 2 (теория) присутствует
  ✅ P.6     [SCRIPT   ] Глава 3 (практика) присутствует

### KEPT (полезные оригинальные)  —  6/6
  ✅ 2.1.1   [SCRIPT   ] Длина аннотации
  ✅ 2.2.2   [SCRIPT   ] Ссылки оглавления ведут на заголовки
  ✅ 2.4.1   [SCRIPT   ] Кол-во частей теории
  ✅ 2.4.3   [SCRIPT   ] Объём каждой части теории
  ✅ 2.5.1   [SCRIPT   ] Кол-во задач практики
  ✅ 2.5.5   [SCRIPT   ] Артефакт и путь в каждой задаче

### FIXED (исправленные)  —  2/3
  ✅ 2.4.6   [SCRIPT   ] Пример/кейс в каждой части (исправлено)
  ❌ 2.4.7   [SCRIPT   ] Читабельность (Флеш-Оборнева, исправлено)
          ↳ средняя 39.2, полоса 45-80
  ✅ 2.4.4   [SCRIPT   ] Определения терминов (перемаркировано) (SOFT

## Before / After — доказательство, что правки работают

In [9]:
print("--- 2.4.7 читабельность ---")
parts = theory_parts(MD)
# СТАРАЯ логика (тавтология)
def old_readability(parts):
    sc=[]
    for _,b in parts:
        raw = oborneva_fre(clean_prose(b)) or 0
        raw_clamped = max(0.0, min(raw, 30.0))
        sc.append(50.0 + 30.0*raw_clamped/30.0)   # всегда в [50,80]
    avg = statistics.mean(sc) if sc else 0
    return avg, (1 if 50<=avg<=80 else 0)
oa, osc = old_readability(parts)
new = next(c for c in REGISTRY if c.id=="2.4.7")
print(f"  СТАРАЯ: avg={oa:.1f} -> score={osc}  (по построению всегда 1)")
print(f"  НОВАЯ : avg={new.details['avg']} (по частям {new.details['per_part_fre']}) -> score={new.score}")

print("\n--- 2.4.6 пример/кейс ---")
fake = "В этой части мы примерно описываем подход. Применяем правила последовательно."
old_hit = any(m in fake.lower() for m in ["пример","ситуация","кейс","случай"])
new_hit = bool(re.search(r"\*\*\s*(Пример|Кейс|Ситуация)", fake, re.I) or
               re.search(r"\b(пример|примеры|кейс|кейсы|ситуация|ситуации)\b", fake.lower()))
print(f"  Текст-ловушка: '{fake}'")
print(f"  СТАРАЯ (substring): {old_hit}  <- ложное срабатывание на 'примерно/применяем'")
print(f"  НОВАЯ (границы слова): {new_hit}  <- корректно: реального примера нет")

--- 2.4.7 читабельность ---
  СТАРАЯ: avg=80.0 -> score=1  (по построению всегда 1)
  НОВАЯ : avg=39.2 (по частям [35.2, 42.9, 39.5]) -> score=0

--- 2.4.6 пример/кейс ---
  Текст-ловушка: 'В этой части мы примерно описываем подход. Применяем правила последовательно.'
  СТАРАЯ (substring): True  <- ложное срабатывание на 'примерно/применяем'
  НОВАЯ (границы слова): False  <- корректно: реального примера нет


## Как читать и что дальше

**Что показал прогон.** Preflight почти весь зелёный (документ структурно собран), часть KEPT/FIXED
прошла, а новые проверки целостности поймали реальные дефекты, которые старые 39 пропускали:
сломанную таблицу, дословные повторы-болванки, диаграммы не по теме, оборванный «Soft skills»».
Before/after доказывает: старая 2.4.7 ставила 1 при любом тексте, новая — реальное число; старый
детектор примеров ловил «примерно», новый — нет.

**Граница ответственности (важно).** Здесь только объективное «есть/нет/целостно». Всё, что про
**качество подачи** (хорош ли пример по сути, связно ли, та ли тональность школы) — это
**дидактическая ось с жюри** из соседнего ноутбука. Не смешивай: структурная ось — гейт «не сломан»,
дидактическая — «хорошо преподано».

**Что доделать:**
1. **Полоса читабельности `(45, 80)` — провизорная.** Откалибруй на 30–50 реальных README по тому,
   что эксперты считают комфортным; сейчас это разумная заглушка, а не истина.
2. **Item-анализ.** Прогони набор и выкинь критерии с нулевой дискриминацией (всегда =1).
3. Перенеси исправленные пороги в `thresholds.py` как единственный источник правды и синхронизируй CRITERIA.md.
4. Подключи HARD-провалы к `MethodologyGate` (стадия evaluation) — поднимать `human_review_required`.

**Запуск на другом документе:** `DOC_PATH=/путь/README.md` и перезапусти. LLM не нужен.
